# Creating the Architecture

In [1]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
# from hellaswag import render_example, iterate_examples

/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def batch_head(x, n_head):
  b,t,c = x.shape
  x = x.reshape(b, t, n_head,-1)
  return x.transpose(1,2).reshape(b*n_head, t, -1)

def head_batch(x, n_head):
    bn, t, c = x.shape
    x = x.reshape(-1, n_head, t, c)
    return x.transpose(1, 2).reshape(-1, t, n_head * c)

In [3]:
@dataclass
class GPT_config:
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 6
    vocab_size: int =  50257 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token
    block_size: int = 1024
    # for dl
    bs : int = 32
    ds_name: str = 'input.txt'
    tokenizer = 'gpt2'

In [4]:
class CasualMultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    assert self.n_embd % config.n_head == 0, "number of heads must be a multiple of n_head"
    self.head_sz = config.n_embd // config.n_head
    self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd)
    self.c_proj = nn.Linear(self.n_embd,self.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
    self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size)))

  def forward(self, x): # x -> (B, T, E)
    b,t,c = x.shape
    x = self.c_attn(x)
    k,q,v = torch.chunk(x, 3, dim=-1) # we get the qkv first before batch_head because this is the order openai used(weights optimized for it), but reversing it when training a model from scratch is fine
    q = batch_head(q, self.n_head) # (B, T, E)
    k = batch_head(k, self.n_head) # (B, T, E)
    v = batch_head(v, self.n_head) # (B, T, E)
    wei = k @ q.transpose(-1,-2) * self.head_sz**-0.5 # Scaling, shape (B, T, T)
    wei = wei.masked_fill(self.bias[:t,:t]==0, float('-inf'))
    wei = F.softmax(wei, -1)
    wei = head_batch(wei @ v, self.n_head)
    return self.c_proj(wei)

In [5]:
m = CasualMultiHeadAttention(GPT_config())
m(torch.rand((4, 1024, 768))).shape

torch.Size([4, 1024, 768])

In [6]:
class MLP(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
    self.gelu    = nn.GELU(approximate='tanh')
    self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
  def forward(self, x):
    return self.c_proj(self.gelu(self.c_fc(x)))

In [7]:
class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.attn = CasualMultiHeadAttention(config)
    self.mlp = MLP(config)
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.ln_2 = nn.LayerNorm(config.n_embd)
  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    return x + self.mlp(self.ln_2(x))

In [8]:
class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(dict(
      wpe = nn.Embedding(config.block_size, config.n_embd),
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd)
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

  def forward(self, x): # x -> (B, Tokens)
    B, T = x.shape
    assert T <= self.config.block_size, f'The Context Exceeds the block size {T} > {self.config.block_size}'
    x = self.transformer.wpe(torch.arange(T, device=x.device)) + self.transformer.wte(x) # x -> (B, tokens, embs)
    for layer in self.transformer.h:
      x = layer(x)
    x = self.transformer.ln_f(x)
    return self.lm_head(x)

In [9]:
m = GPT(GPT_config(vocab_size=32)) # smaller vocab_size for debugging
m(torch.randint(0,32, (4, 512))).shape

torch.Size([4, 512, 32])

# Loading Pretrained Model & Sampler from it

In [10]:
class GPT_V2(GPT):
    def __init__(self, config):
        super().__init__(config)

    @classmethod
    def from_pretrained(cls, model_name): #  cls -> the class it self can be called -> cls()
        assert model_name in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}, f"{model_name} is not Valid" # make a set so that it is faster O(1)
        from transformers import GPT2LMHeadModel
        print(f'loading {model_name}')
        configs ={
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_name]
        configs['vocab_size'] = 50257 # same in all of them
        configs['block_size'] = 1024 # same in all of them

        model_config = GPT_config(**configs)
        model = cls(model_config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        model_hf = GPT2LMHeadModel.from_pretrained(model_name) # load the hugging face model
        sd_hf = model_hf.state_dict()
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight'] # openai checkpoints use a "Conv1D" module, but we only want to use a  Linear so we have to transpose these weights when we import them

        assert len(sd_keys) == len(sd_keys_hf), f'Keys mismatch : {len(sd_keys_hf)} != {len(sd_keys)}'
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape, f'shape mismatch Transposed if Conv-> {sd_hf[k].shape} != {sd[k].shape} at {k}'   # [::-1] reverse sequence e.g. (768, 3072) to (3072, 768)
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].T)
            else:
                assert sd_hf[k].shape == sd[k].shape , f'shape mismatch {sd_hf[k].shape} != {sd[k].shape} at {k}'
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])


        return model

- `**config_args` unpacks the dict into keyword arguments — same as writing `GPTConfig(n_layer=12, n_head=12, n_embd=768, vocab_size=50257, block_size=1024)`
- if `*config_args` it only gives `n_layer, n_head,....`

    sd = model.state_dict()
    state_dict() is a built-in PyTorch method on every nn.Module. Returns an ordered dict of every parameter in the model by name:
            {
                'transformer.wte.weight': tensor([[...]]),   # embedding weights
                'transformer.h.0.ln_1.weight': tensor([...]), # layer norm weights
                ...
            }
    sd_keys = sd.keys()
    Just the names, not the tensors themselves. Like:
    ['transformer.wte.weight', 'transformer.h.0.ln_1.weight', ...]

In [11]:
pre_trained_model = GPT_V2.from_pretrained('gpt2')
print("works")

loading gpt2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

works


In [12]:
# Model Generation (from the pretraining)
num_sequence = 5
max_length = 30

In [13]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode('I am pharmacist')
tokens

[40, 716, 10113, 20279]

In [14]:
tokens = torch.tensor(tokens, dtype=torch.long)
tokens, tokens.shape

(tensor([   40,   716, 10113, 20279]), torch.Size([4]))

In [15]:
tokens = tokens.repeat(num_sequence,1)
tokens, tokens.shape

(tensor([[   40,   716, 10113, 20279],
         [   40,   716, 10113, 20279],
         [   40,   716, 10113, 20279],
         [   40,   716, 10113, 20279],
         [   40,   716, 10113, 20279]]),
 torch.Size([5, 4]))

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokens = tokens.to(device)

In [17]:
tokens.size(1), tokens.shape

(4, torch.Size([5, 4]))

In [18]:
pre_trained_model = pre_trained_model.to(device)

In [19]:
# visualize the topk
with torch.no_grad():
    logits = pre_trained_model(tokens)
    logits = logits[:,-1,:]
    probs = F.softmax(logits, dim=-1)
    topk_probs, topk_indices = torch.topk(probs, k=5, dim=-1) # take the highest 5 probs
topk_probs.shape, topk_indices.shape, topk_probs[0], topk_indices[0]

(torch.Size([5, 5]),
 torch.Size([5, 5]),
 tensor([0.1931, 0.1761, 0.1619, 0.0439, 0.0390], device='cuda:0'),
 tensor([ 13,  11, 290, 379, 329], device='cuda:0'))

In [20]:
while tokens.shape[1] < max_length:
    with torch.no_grad():
        logits = pre_trained_model(tokens)
        logits = logits[:,-1,:]
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, k=50, dim=-1) # take the highest 50 probs
        ix = torch.multinomial(topk_probs, num_samples=1)  # take one sample
        ids = torch.gather(topk_indices, dim=-1, index=ix)
        tokens = torch.cat((tokens, ids), dim=1)

In [21]:
for i in range(num_sequence):
    decoded = enc.decode(tokens[i].tolist())
    print(f'sample{i+1}: {decoded}')

sample1: I am pharmacist. I work with an old friend of mine in my position as an employee, on research into cancer, and we do try to
sample2: I am pharmacist. I am the father of three children. What can I do because there is no one in the hospital right now. How can
sample3: I am pharmacist here. I use only the finest medicines and only I like to use the best. This pharmacy is very friendly and I would recommend
sample4: I am pharmacist. She has become a nurse and is our next doctor, so we can talk about what kind of research is going on, what
sample5: I am pharmacist. You need to be ready."

He walked into a local drugstore in New Albany, where drugstores that sell pain


# Get Data & Work on the Model from scratch

In [22]:
model = GPT(GPT_config())

In [23]:
# Get the tiny shakespeare Dataset
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [24]:
with open('input.txt', 'r') as f:
    txt = f.read()
print(txt[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [25]:
# tokenize
all_tokens = enc.encode(txt)
all_tokens = torch.tensor(all_tokens, dtype=torch.long)
all_tokens.shape

torch.Size([338025])

In [26]:
# make Some Dl
B, T = 5, 6 # 5 so that it is compatible with 338025
tokens = all_tokens[:B*T+1] # +1 so that we have a label for all input tokens
x_tokens = tokens[:-1].view(-1, T) # [:-1] -> get all the except the last one
y_tokens = tokens[1:].view(-1, T) # -1 is B
x_tokens.shape, y_tokens.shape, x_tokens, y_tokens

(torch.Size([5, 6]),
 torch.Size([5, 6]),
 tensor([[ 5962, 22307,    25,   198,  8421,   356],
         [ 5120,   597,  2252,    11,  3285,   502],
         [ 2740,    13,   198,   198,  3237,    25],
         [  198,  5248,   461,    11,  2740,    13],
         [  198,   198,  5962, 22307,    25,   198]]),
 tensor([[22307,    25,   198,  8421,   356,  5120],
         [  597,  2252,    11,  3285,   502,  2740],
         [   13,   198,   198,  3237,    25,   198],
         [ 5248,   461,    11,  2740,    13,   198],
         [  198,  5962, 22307,    25,   198,  1639]]))

In [27]:
with torch.no_grad():
    logits = model(x_tokens)
    logits.shape

In [28]:
class GPT_V3(GPT_V2):
  def forward(self, x, targets=None): # x -> (B, Tokens)
    B, T = x.shape
    assert T <= self.config.block_size, f'The Context Exceeds the block size {T} > {self.config.block_size}'
    x = self.transformer.wpe(torch.arange(T, device=x.device)) + self.transformer.wte(x) # x -> (B, tokens, embs)
    for layer in self.transformer.h:
      x = layer(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    if targets is not None:
        loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss
    return logits

In [29]:
# Optimization loop
model = GPT_V3(GPT_config())
optim = torch.optim.AdamW(model.parameters(), lr=5e-3)
steps = 5
for i in range(steps):
    logits = model(x_tokens)
    loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), y_tokens.view(-1))
    print(f"loss at {i}: {loss.item()}")
    loss.backward()
    optim.step()
    optim.zero_grad()

loss at 0: 10.971329689025879
loss at 1: 3.850935459136963
loss at 2: 4.905251979827881
loss at 3: 8.647820472717285
loss at 4: 3.6836602687835693


In [30]:
# Optimization loop (using the forward implementation we made)
model = GPT_V3(GPT_config())
optim = torch.optim.AdamW(model.parameters(), lr=5e-3)
steps = 5
for i in range(steps):
    logits, loss = model(x_tokens, y_tokens)
    print(f"loss at {i}: {loss.item()}")
    loss.backward()
    optim.step()
    optim.zero_grad()

loss at 0: 11.03085994720459
loss at 1: 3.5852627754211426
loss at 2: 5.117702007293701
loss at 3: 7.659849643707275
loss at 4: 3.3518033027648926


# Model Intialization, etc....

In [31]:
class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(dict(
      wpe = nn.Embedding(config.block_size, config.n_embd),
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd)
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    ## MAKE the last Linear later the same as the embedding layer
    self.lm_head.weight = self.transformer.wte.weight

    ## APPLY Initialization
    self.apply(self._initalization)

  ## Make Initialization function
  def _initalization(self, module):
      if isinstance(module, nn.Linear):
          std = 0.02
          if hasattr(module, 'INIT_SPECIAL_STD'):
              std *= (2 * self.config.n_layer) ** -0.5
          nn.init.normal_(module.weight,mean=0., std=std) # torch.nn
          if module.bias is not None:
              nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          nn.init.normal_(module.weight, mean=0., std=0.02)

  def forward(self, x, targets=None): # x -> (B, Tokens)
    B, T = x.shape
    assert T <= self.config.block_size, f'The Context Exceeds the block size {T} > {self.config.block_size}'
    x = self.transformer.wpe(torch.arange(T, device=x.device)) + self.transformer.wte(x) # x -> (B, tokens, embs)
    for layer in self.transformer.h:
      x = layer(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    if targets is not None:
        loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss
    return logits

  @classmethod
  def from_pretrained(cls, model_name): #  cls -> the class it self can be called -> cls()
    assert model_name in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}, f"{model_name} is not Valid" # make a set so that it is faster O(1)
    from transformers import GPT2LMHeadModel
    print(f'loading {model_name}')
    configs ={
        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
    }[model_name]
    configs['vocab_size'] = 50257 # same in all of them
    configs['block_size'] = 1024 # same in all of them

    model_config = GPT_config(**configs)
    model = cls(model_config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

    model_hf = GPT2LMHeadModel.from_pretrained(model_name) # load the hugging face model
    sd_hf = model_hf.state_dict()
    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight'] # openai checkpoints use a "Conv1D" module, but we only want to use a  Linear so we have to transpose these weights when we import them

    assert len(sd_keys) == len(sd_keys_hf), f'Keys mismatch : {len(sd_keys_hf)} != {len(sd_keys)}'
    for k in sd_keys_hf:
        if any(k.endswith(w) for w in transposed):
            assert sd_hf[k].shape[::-1] == sd[k].shape, f'shape mismatch Transposed if Conv-> {sd_hf[k].shape} != {sd[k].shape} at {k}'   # [::-1] reverse sequence e.g. (768, 3072) to (3072, 768)
            with torch.no_grad():
                sd[k].copy_(sd_hf[k].T)
        else:
            assert sd_hf[k].shape == sd[k].shape , f'shape mismatch {sd_hf[k].shape} != {sd[k].shape} at {k}'
            with torch.no_grad():
                sd[k].copy_(sd_hf[k])

    return model

In [32]:
# test it
m = GPT(GPT_config())

# Making Flash attention

In [33]:
class CasualMultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    assert self.n_embd % config.n_head == 0, "number of heads must be a multiple of n_head"
    self.head_sz = config.n_embd // config.n_head
    self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd)
    self.c_proj = nn.Linear(self.n_embd,self.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
    self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size)))

  def forward(self, x): # x -> (B, T, E)
    b,t,c = x.shape
    x = self.c_attn(x)
    k,q,v = torch.chunk(x, 3, dim=-1) # we get the qkv first before batch_head because this is the order openai used(weights optimized for it), but reversing it when training a model from scratch is fine
    q = batch_head(q, self.n_head) # (B, T, E)
    k = batch_head(k, self.n_head) # (B, T, E)
    v = batch_head(v, self.n_head) # (B, T, E)
    # wei = k @ q.transpose(-1,-2) * self.head_sz**-0.5 # Scaling, shape (B, T, T)
    # wei = wei.masked_fill(self.bias[:t,:t]==0, float('-inf'))
    # wei = F.softmax(wei, -1)
    # wei = wei @ v
    wei = F.scaled_dot_product_attention(q, k, v, is_causal=True) # is_casual=True -> causal mask is the triangular mask we built manually with torch.tril + masked_fill.
    wei = head_batch(wei, self.n_head)
    return self.c_proj(wei)

In [34]:
# test it
m = GPT(GPT_config(vocab_size=32))
m(torch.randint(0,31,(4,8))).shape

torch.Size([4, 8, 32])

# Create a more detailed GPT_config

In [56]:
# make all hyper params here to make it easier
@dataclass
class GPT_config:
    ## for model
    n_embd: int = 64
    n_head: int = 4
    n_layer: int = 4
    vocab_size: int =  50257
    block_size: int = 32
    ## for data
    bs : int = 32
    ds_name: str = 'input.txt'
    tokenizer: str = 'gpt2'
    tot_bs_for_grad_accum: int = 0
    # for the training shards
    split: str = 'Train'
    data_root:str = "edu_fineweb10B"
    ## for optim sched
    max_lr: float = 6e-4
    min_lr_ratio: float = 0.1
    warmup_steps: int = 10
    max_steps: int = 50
    ## for optim
    beta1: float = 0.9
    beta2: float = 0.95
    eps: float = 1e-8
    weight_decay: float = 0.1
    ## device
    device = device
    ## Multi-Device Training
    process_rank = ddp_rank
    num_processes = ddp_world_size

In [57]:
config = GPT_config()

# LR shced

In [37]:
# Creating a LR sched
def get_lr(step, config):
    min_lr = config.max_lr * config.min_lr_ratio
    if step < config.warmup_steps:
        return config.max_lr/config.warmup_steps * (step+1)
    if step > config.max_steps :
        return min_lr
    decay_ratio = (step-config.warmup_steps)/(config.max_steps-config.warmup_steps) # make a ratio between 0 and 1 that represent the current section in the cos
    assert 0 <= decay_ratio <= 1, f'There is something wrong with max steps:{config.max_steps}, step:{step}, warmup_steps:{config.warmup_steps}'
    coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (config.max_lr - min_lr)

    At decay_ratio=0 → cos(0) = 1 → coeff = 0.5 * (1 + 1) = 1.0
    At decay_ratio=1 → cos(π) = -1 → coeff = 0.5 * (1 - 1) = 0.0

In [38]:
# test it
for i in range(0,101,10):
    print(f'At {i} lr is : {get_lr(i, config)}')

At 0 lr is : 5.9999999999999995e-05
At 10 lr is : 0.0005999999999999998
At 20 lr is : 0.0005209188309203677
At 30 lr is : 0.00032999999999999994
At 40 lr is : 0.00013908116907963218
At 50 lr is : 5.9999999999999995e-05
At 60 lr is : 5.9999999999999995e-05
At 70 lr is : 5.9999999999999995e-05
At 80 lr is : 5.9999999999999995e-05
At 90 lr is : 5.9999999999999995e-05
At 100 lr is : 5.9999999999999995e-05


# Optimizers configs (wd,...)

In [39]:
# Creating optimizer that is going to apply the weight decay only to certain weights
class GPT_V4(GPT):
  def optimizers_config(self):
      # Take all the requires_grad parameters - this 2 lines can be collapsed into one line but we make it in 2 for clearner steps -
      params_dict = {pn: p for pn, p in self.named_parameters()}
      params_dict = {pn: p for pn, p in params_dict.items() if p.requires_grad}
      # Decay only when <= 2 dims, dims < 2 are the 1D tensors -> biases and LayerNorm's, It doesn't make sense to decay them
      decay_params = [p for pn, p in params_dict.items() if p.dim() >= 2]
      no_decay_params = [p for pn, p in params_dict.items() if p.dim() < 2]
      # create optim group
      optim_group = [
          {'params': decay_params, 'weight_decay': self.config.weight_decay},
          {'params': no_decay_params, 'weight_decay': 0.},
      ]
      # count the params of each group
      num_decay_params = sum(p.numel() for p in decay_params)
      num_no_decay_params = sum(p.numel() for p in no_decay_params)
      # Print it
      if master_process:
          print(f'Number of the decayed Tensors : {len(decay_params)} |  Number of the decayed params : {num_decay_params:,}, ')
          print(f'Number of the decayed Tensors : {len(no_decay_params)} |  Number of no_decay_param: {num_no_decay_params:,}')
      # See if we can make fused Adam
      fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
      use_fuse = fused_available and 'cuda' in self.config.device
      if master_process:
          print(f'Using fused AdamW: {use_fuse}')
      # Make the optim and also add the Betas and the eps that openai used in the training
      optimizer = torch.optim.AdamW(optim_group, lr=self.config.max_lr, betas=(self.config.beta1, self.config.beta2), eps=self.config.eps, fused=use_fuse)
      return optimizer

    named_parameters() returns a GENERATOR (one-time-use iterator)
        → can't call .items() on it
        → can only loop over it once

    param_dict = {pn: p for pn, p in self.named_parameters()}
        → converts generator into a real dict
        → now .items() works
        → can loop over it as many times as needed

    ┌────────────────────┬────────────────────────┬──────────────────────┬─────────┐
    │                     │ Parameters (trainable) │ Parameters (frozen)  │ Buffers │
    ├────────────────────┼────────────────────────┼──────────────────────┼─────────┤
    │ named_parameters()  │ Yes                    │ Yes                  │ No      │
    │ state_dict()        │ Yes                    │ Yes                  │ Yes     │
    └────────────────────┴────────────────────────┴──────────────────────┴─────────┘

    nn.Linear(768, 768)  →  two tensors: weight + bias

    weight: shape (768, 768)
    bias:   shape (768,)

    len([weight, bias])              → 2         (just counts: 2 tensors total)

    sum(p.numel() for p in [weight, bias])
      → (768*768) + 768
      → 589,824 + 768
      → 590,592     (total individual values across both tensors)

    code >>>:        inspect.signature(torch.optim.AdamW)

    Inspects the AdamW class/function and returns an object describing its accepted parameters.

    code >>>:         inspect.signature(torch.optim.AdamW).parameters

    Returns a dict-like collection of all parameter names AdamW.__init__ accepts:
    {'params': ..., 'lr': ..., 'betas': ..., 'eps': ..., 'weight_decay': ..., 'fused': ..., ...}

    code >>>:         'fused' in inspect.signature(torch.optim.AdamW).parameters

    Checks if 'fused' is one of those parameter names — True if this PyTorch version supports it, False if it's an older version that doesn't have that option yet.

In [40]:
model = GPT_V4(config)

In [41]:
optimizer = model.optimizers_config()

Number of the decayed Tensors : 18 |  Number of the decayed params : 3,415,104, 
Number of the decayed Tensors : 34 |  Number of no_decay_param: 3,456
Using fused AdamW: True


# Making DL_Lite

In [65]:
class DL_lite:
    def __init__(self, config):
        # store attr
        self.block_size= config.block_size
        self.process_rank = config.process_rank
        self.num_processes = config.num_processes
        self.bs = config.bs
        assert config.tot_bs_for_grad_accum % self.bs == 0, f' Total_Batch_size: {config.tot_bs_for_grad_accum} is not divisible by Batch_size: {self.bs} * Num_processes: {self.num_processes}'
        grad_accum = bool(config.tot_bs_for_grad_accum)
        self.tot_mini_batches = config.tot_bs_for_grad_accum // (self.bs * self.num_processes) if grad_accum else 1
        # read file
        with open(config.ds_name, 'r') as f:
            self.txt = f.read()
        if master_process:
            print(f'loading file : {config.ds_name}')
        # make the encoder
        self.enc = tiktoken.get_encoding(config.tokenizer)
        # encode Data
        self.tokens = torch.tensor(self.enc.encode(self.txt), dtype=torch.long)
        if master_process:
            print(f'Number Tokens is = {len(self.tokens)} Tokens | Total Batch Size = {config.tot_bs_for_grad_accum} | Grad Accum = {grad_accum} | Number Mini Batches = {self.tot_mini_batches} | Num_processes: {self.num_processes}')
        # Make a tracker
        self.tr = self.bs * self.block_size * self.process_rank
    def after_batch(self):
        tokens = self.tokens[self.tr : self.tr + self.bs * self.block_size + 1 ]
        self.tr += self.bs * self.block_size * self.num_processes
        x = tokens[:-1].view(self.bs, -1)
        y = tokens[1:].view(self.bs, -1)
        if (self.bs * self.block_size * self.num_processes  + 1 + self.tr) > len(self.tokens):
            self.tr = self.bs * self.block_size * self.process_rank
        return x, y

dl = DL_lite(GPT_config(vocab_size=32, tot_bs_for_grad_accum=64))

loading file : input.txt
Number Tokens is = 338025 batches | Total Batch Size = 64 | Grad Accum = True | Number Mini Batches = 2 | Num_processes: 1


In [67]:
# dl = DL_lite(GPT_config(vocab_size=32, tot_bs_for_grad_accum=63)) # Testing the error

In [44]:
for i in range(5):
    x,y = dl.after_batch()
    print(f'x: {x.shape}, y:{y.shape}')

x: torch.Size([32, 32]), y:torch.Size([32, 32])
x: torch.Size([32, 32]), y:torch.Size([32, 32])
x: torch.Size([32, 32]), y:torch.Size([32, 32])
x: torch.Size([32, 32]), y:torch.Size([32, 32])
x: torch.Size([32, 32]), y:torch.Size([32, 32])


# Create a Optimization loop

In [45]:
# instance
from functools import partial
config = GPT_config(tot_bs_for_grad_accum=64)
model = GPT_V4(config)
model.to(config.device)
lr_getter = partial(get_lr, config=config)
dl = DL_lite(config)
optimizer = model.optimizers_config()

loading file : input.txt
1 epochs is = 544 batches | Grad Accum = True
Number of the decayed Tensors : 18 |  Number of the decayed params : 3,415,104, 
Number of the decayed Tensors : 34 |  Number of no_decay_param: 3,456
Using fused AdamW: True


In [46]:
# Training Loop
for step in range(200):
    start = time.time()
    accum_loss = 0
    for mini_step in range(dl.tot_mini_batches):
        x,y = dl.after_batch()
        x,y = x.to(config.device), y.to(config.device)
        with torch.autocast(device_type=config.device, dtype=torch.bfloat16):
            logits, loss = model(x, y)
            loss /= dl.tot_mini_batches
            accum_loss += loss
        if ddp:
            model.require_backward_grad_sync = (mini_step == dl.tot_mini_batches - 1) # So that we avoid unnecessary communication during backward () unless it is the last step
        loss.backward()
    if ddp:
        dist.all_reduce(accum_loss, op= dist.ReduceOp.AVG) # Averaging the Loss Across all the Processes
    lr = lr_getter(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.) # This Makes Normalization for the Norm of the Gradients proportionally, so that -> root(sum(grads**2)) <= 1
    optimizer.step()
    optimizer.zero_grad()
    if config.device == 'cuda' : torch.cuda.synchronize() # so that the cpu don't run the next command while the GPU still hasn't Finished
    end = time.time()
    time_taken = end - start
    tokens_count = dl.block_size * dl.bs * dl.tot_mini_batches
    if master_process:
        print(f'Step: {step:5d} | Loss: {accum_loss.item():.4f} | lr = {lr:.6f} | Norm = {norm:3f} | Time: {time_taken:.4f}sec | Token/sec: {(tokens_count / time_taken):.3f}')

if ddp:
    destroy_process_group() # Clean After the Multi-GPU Process

Step: 0 | Loss: 10.8163 | lr = 0.000060 | Norm = 2.512606 | Time: 0.1410sec | Token/sec: 14526.490
Step: 1 | Loss: 10.8065 | lr = 0.000120 | Norm = 2.552736 | Time: 0.0336sec | Token/sec: 60979.474
Step: 2 | Loss: 10.7829 | lr = 0.000180 | Norm = 2.527033 | Time: 0.0295sec | Token/sec: 69359.246
Step: 3 | Loss: 10.7435 | lr = 0.000240 | Norm = 2.360540 | Time: 0.0301sec | Token/sec: 68145.960
Step: 4 | Loss: 10.7018 | lr = 0.000300 | Norm = 1.931514 | Time: 0.0311sec | Token/sec: 65841.921
Step: 5 | Loss: 10.6499 | lr = 0.000360 | Norm = 1.399809 | Time: 0.0324sec | Token/sec: 63142.248
Step: 6 | Loss: 10.5950 | lr = 0.000420 | Norm = 1.277597 | Time: 0.0300sec | Token/sec: 68251.544
Step: 7 | Loss: 10.5605 | lr = 0.000480 | Norm = 1.210659 | Time: 0.0300sec | Token/sec: 68319.398
Step: 8 | Loss: 10.5148 | lr = 0.000540 | Norm = 1.266257 | Time: 0.0393sec | Token/sec: 52089.253
Step: 9 | Loss: 10.4760 | lr = 0.000600 | Norm = 1.308753 | Time: 0.0372sec | Token/sec: 54981.211
Step: 10 |

# Distributed Training

In [68]:
# Creating distributed training
from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    assert torch.cuda.is_available(), f'There is not cude for the device to run ddp'
    init_process_group(backend='nccl')
    ddp_rank = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])
    device = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
else:
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1
    master_process = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')

Using device: cuda


# Adjust DL_lite

In [69]:
import numpy as np
def load_tokens(filename):
    np_loaded =   np.load(filename)
    return torch.tensor(np_loaded, dtype=torch.long)

In [ ]:
class DL_lite:
    def __init__(self, config):
        assert config.split in {'Train', 'Val'}, 'Split must be one of "Train" or "Val"'
        # store attr
        self.block_size= config.block_size
        self.process_rank = config.process_rank
        self.num_processes = config.num_processes
        self.bs = config.bs
        assert config.tot_bs_for_grad_accum % self.bs == 0, f' Total_Batch_size: {config.tot_bs_for_grad_accum} is not divisible by Batch_size: {self.bs} * Num_processes: {self.num_processes}'
        grad_accum = bool(config.tot_bs_for_grad_accum)
        self.tot_mini_batches = config.tot_bs_for_grad_accum // (self.bs * self.num_processes) if grad_accum else 1
        # Loading the data
        data_root = config.data_root
        shards = os.listdir(data_root)
        shards = [s for s in shards if config.split in s]
        shards = sorted(shards)
        self.shards = [os.path.join(data_root, s) for s in shards]
        assert len(shards) > 0, f"no shards found for split {config.split}"
        if master_process:
            print(f'Found Shards =  {len(self.shards)} | Total Batch Size = {config.tot_bs_for_grad_accum} | Grad Accum = {grad_accum} | Number Mini Batches = {self.tot_mini_batches} | Num_processes: {self.num_processes}')
        # Make a trackers
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.tr = self.bs * self.block_size * self.process_rank
    def after_batch(self):
        tokens = self.tokens[self.tr : self.tr + self.bs * self.block_size + 1 ]
        self.tr += self.bs * self.block_size * self.num_processes
        x = tokens[:-1].view(self.bs, -1)
        y = tokens[1:].view(self.bs, -1)
        if (self.bs * self.block_size * self.num_processes  + 1 + self.tr) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards) # So that we advance to the next shard and if we finish the shards we loop again because of -> %
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.tr = self.bs * self.block_size * self.process_rank
        return x, y

dl = DL_lite(GPT_config(vocab_size=32, tot_bs_for_grad_accum=64))

    FOR THIS CODE:
    # Loading the data
    data_root = config.data_root
    shards = os.listdir(data_root)
    shards = [s for s in shards if config.split in s]
    shards = sorted(shards)
    self.shards = [os.path.join(data_root, s) for s in shards]


------------------------------------

    Let's say edu_fineweb10B/ contains these files:
    train_000_00000.npy
    train_001_00000.npy
    val_000_00000.npy
    val_001_00000.npy

    Step 1: os.listdir(data_root)
    pythonshards = ['val_001_00000.npy', 'train_000_00000.npy', 'val_000_00000.npy', 'train_001_00000.npy']
    Just dumps all filenames in random order, no structure.

    Step 2: [s for s in shards if split in s]
    If split = 'train', the string 'train' is checked against each filename. 'train' in 'train_000_00000.npy' is True. 'train' in 'val_000_00000.npy' is False. So you get:
    pythonshards = ['train_000_00000.npy', 'train_001_00000.npy']

    Step 3: sorted(shards)
    Strings sort alphabetically in Python, and since your filenames have zero-padded numbers (000, 001), alphabetical order is the same as numerical order:
    pythonshards = ['train_000_00000.npy', 'train_001_00000.npy']
    This matters because you want shard 0 before shard 1. Without sorting, os.listdir gives random filesystem order.

    Step 4: [os.path.join(data_root, s) for s in shards]
    os.path.join just glues the folder and filename together with the right separator:
    pythonshards = ['edu_fineweb10B/train_000_00000.npy', 'edu_fineweb10B/train_001_00000.npy']
    Now you have full paths you can actually open with numpy.load() or torch.tensor().

In [2]:
import torch
x = torch.arange(8)
x

/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [4]:
x.repeat(2)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 0, 1, 2, 3, 4, 5, 6, 7])

In [9]:
torch.zeros(()) == torch.zeros(1)

tensor([True])

    prev = F.pad(self.smear_lambda * gate * x[:,:-1], (0, 0, 1, 0))
    #                                                   ↑  ↑  ↑  ↑
    #                                              last_L last_R  T_L  T_R